# Constructing a dataset

In [1]:
import pandas as pd
import numpy as np

In [44]:
# load datasets
df_23 = pd.read_csv("data/raw/climate_misinfo_2023.csv")
df_24 = pd.read_csv("data/raw/climate_misinfo_2024.csv")
df_25 = pd.read_csv("data/raw/climate_misinfo_2025.csv")

In [45]:
# merge the datasets
df = pd.concat([df_23, df_24, df_25], ignore_index=True)

df.shape

(182584, 21)

In [46]:
# save full dataframe
df.to_csv("data/processed/climate_misinfo.csv", index=False)

## Create a stratified dataset for annotation testing

In [6]:
df = pd.read_csv("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/climate_misinfo.csv")

pilot_sample = pd.read_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/pilot_sample.parquet")

In [9]:
print("Full dataset:", len(df))
print("Previous pilot:", len(pilot_sample))

Full dataset: 182584
Previous pilot: 10068


In [10]:
pilot_keys = set(pilot_sample["ArticleKey"])

df = df[~df["ArticleKey"].isin(pilot_keys)].copy()

print("Remaining pool:", len(remaining))

Remaining pool: 172516


In [14]:
# set size of subset
TOTAL_SAMPLE = 8500
RANDOM_STATE = 42

# compute year distribution
year_props = df["Year"].value_counts(normalize=True)

# compute target size per year
year_targets = (year_props * TOTAL_SAMPLE).round().astype(int)

sampled_list = []

In [15]:
# sample data
for year, year_n in year_targets.items():
    
    df_year = df[df["Year"] == year]
    
    # proportion per media within that year
    media_props = df_year["Media"].value_counts(normalize=True)
    media_targets = (media_props * year_n).round().astype(int)
    
    for media, media_n in media_targets.items():
        df_media = df_year[df_year["Media"] == media]
        
        # Undgå at sample mere end findes
        media_n = min(media_n, len(df_media))
        
        sampled_media = df_media.sample(
            n=media_n,
            random_state=RANDOM_STATE
        )
        
        sampled_list.append(sampled_media)

final_sample = pd.concat(sampled_list).reset_index(drop=True)

print("Final sample size:", final_sample.shape)
print(final_sample["Year"].value_counts(normalize=True))

Final sample size: (7935, 21)
Year
2023    0.417517
2024    0.323125
2025    0.259357
Name: proportion, dtype: float64


In [16]:
# check media distribution of sample
final_sample["Media"].value_counts(normalize=True)

Media
Ritzaus Bureau       0.025457
Fyens.dk             0.016635
Børsen.dk            0.015501
Berlingske.dk        0.014619
Altinget.dk          0.014367
                       ...   
Ditry.dk             0.000126
Fiskeritidende.dk    0.000126
Version2.dk          0.000126
Morningstar.dk       0.000126
Holstebro.dk         0.000126
Name: proportion, Length: 685, dtype: float64

In [17]:
print("Count os unique media:", len(final_sample["Media"].unique()))

Count os unique media: 685


In [18]:
# save dataframe
final_sample.to_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/pilot_sample_batch2.parquet", index=False)